In [2]:
import pandas as pd
import yfinance as yf
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load your saved news
news = pd.read_csv("../data/all_news.csv")
news['date'] = pd.to_datetime(news['date'])
news['date'] = news['date'].dt.date  # strip time, keep only date

# Get stock prices
stock = yf.download("RELIANCE.NS", period="1mo", auto_adjust=True)
stock = stock[['Close']].reset_index()
stock.columns = ['date', 'close']
stock['date'] = pd.to_datetime(stock['date']).dt.date

# Calculate next day return
stock['next_day_return'] = stock['close'].pct_change().shift(-1)
stock['price_moved'] = stock['next_day_return'].apply(
    lambda x: 'UP' if x > 0 else 'DOWN'
)

# Score each headline
analyzer = SentimentIntensityAnalyzer()
news['score'] = news['headline'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

# Average sentiment per day
daily_sentiment = news.groupby('date')['score'].mean().reset_index()
daily_sentiment.columns = ['date', 'avg_sentiment']

# Merge
merged = pd.merge(stock, daily_sentiment, on='date', how='inner')
print(merged)
merged.to_csv("../data/merged_data.csv", index=False)

[*********************100%***********************]  1 of 1 completed

          date        close  next_day_return price_moved  avg_sentiment
0   2026-02-16  1437.099976        -0.009811        DOWN       0.427378
1   2026-02-17  1423.000000         0.012860          UP       0.225900
2   2026-02-18  1441.300049        -0.022063        DOWN       0.387733
3   2026-02-19  1409.500000         0.007024          UP       0.210500
4   2026-02-20  1419.400024         0.006059          UP      -0.246960
5   2026-02-23  1428.000000         0.000560          UP      -0.183560
6   2026-02-24  1428.800049        -0.021207        DOWN      -0.215377
7   2026-02-25  1398.500000         0.005935          UP       0.017233
8   2026-02-26  1406.800049        -0.009170        DOWN       0.344880
9   2026-02-27  1393.900024        -0.025755        DOWN      -0.306283
10  2026-03-02  1358.000000        -0.009573        DOWN      -0.173500
11  2026-03-04  1345.000000         0.033011          UP      -0.003636
12  2026-03-05  1389.400024         0.011084          UP       0